## set up

In [41]:
import pandas as pd
import glob
import os
from itables import show

# initial data

In [42]:
books_before = pd.read_csv('./data/processed/books_after_cleaning_4.csv')
books_before.info()

<class 'pandas.DataFrame'>
RangeIndex: 29770 entries, 0 to 29769
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   kode_buku      29134 non-null  str    
 1   isbn_issn      26489 non-null  str    
 2   judul_buku     29770 non-null  str    
 3   subjudul       10587 non-null  str    
 4   penulis        28449 non-null  str    
 5   nama_belakang  28449 non-null  str    
 6   topik          29770 non-null  str    
 7   deskripsi      229 non-null    str    
 8   bahasa         29770 non-null  str    
 9   kategori       29770 non-null  str    
 10  tahun_terbit   29088 non-null  float64
dtypes: float64(1), str(10)
memory usage: 7.0 MB


In [43]:
books_before['deskripsi'].notna().mean() * 100

np.float64(0.7692307692307693)

In [44]:
books_before[books_before['judul_buku'] == 'administrasi pemerintahan daerah']

,kode_buku,isbn_issn,judul_buku,subjudul,penulis,nama_belakang,topik,deskripsi,bahasa,kategori,tahun_terbit
17895,"['91243501', '91243502', '91243503']",9789790114203,administrasi pemerintahan daerah,NaN,Hanif Nurcholis,nurcholis,['public administration'],NaN,Indonesia,Ilmu Sosial,2017.0


# Hasil Enrich

### dari sqlite

In [45]:
import sqlite3
import json
DB_FILE = f"google_books_cache_v3.db"
conn = sqlite3.connect(DB_FILE)
cursor = conn.cursor()

def get_all_cached_data():
    conn = sqlite3.connect(DB_FILE)
    df_raw = pd.read_sql_query("SELECT * FROM books_cache", conn)
    conn.close()
    return pd.DataFrame(df_raw)

In [46]:
enrich_df = get_all_cached_data()
enrich_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30076 entries, 0 to 30075
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   key          30076 non-null  str    
 1   title        30076 non-null  str    
 2   author       30076 non-null  str    
 3   raw_isbn     26623 non-null  str    
 4   query        30076 non-null  str    
 5   confidence   30076 non-null  float64
 6   volume_info  22852 non-null  str    
 7   timestamp    30076 non-null  str    
dtypes: float64(1), str(7)
memory usage: 43.0 MB


In [47]:
enrich_df.sample(5)

,key,title,author,raw_isbn,query,confidence,volume_info,timestamp
10759,advanced concepts in physical chemistry|ernest...,advanced concepts in physical chemistry,ernest d. kaufman,NaN,advanced%20concepts%20in%20physical%20chemistr...,1.0,"{""title"": ""Advanced Concepts in Physical Chemi...",2026-02-24 15:21:29
28153,"sosiologi, jilid 1|paul b. horton|nan","sosiologi, jilid 1",paul b. horton,NaN,sosiologi%2C%20jilid%201+inauthor:paul%20b.%20...,0.0,NaN,2026-03-01 18:17:00
690,management information systems|raymond mcleod|...,management information systems,raymond mcleod,0130315494,isbn:0130315494,1.0,"{""title"": ""Management Information Systems"", ""a...",2026-02-22 20:42:04
25280,statistik untuk pemimpin berwawasan global|j.s...,statistik untuk pemimpin berwawasan global,j.supranto,9796910527,statistik%20untuk%20pemimpin%20berwawasan%20gl...,0.0,NaN,2026-03-01 16:44:17
13089,adapting to a changing colorado river|david g....,adapting to a changing colorado river,"david g. groves, jordan r. fischbach, evan blo...",9780833084811,isbn:9780833084811,1.0,"{""title"": ""Adapting to a Changing Colorado Riv...",2026-02-24 16:38:51


In [48]:
enrich_df[enrich_df['raw_isbn'] == '9789790114203']

,key,title,author,raw_isbn,query,confidence,volume_info,timestamp
18206,administrasi pemerintahan daerah|hanif nurchol...,administrasi pemerintahan daerah,hanif nurcholis,9789790114203,isbn:9789790114203,1.0,"{""title"": ""Administrasi pemerintahan daerah"",\...",2026-02-28 20:49:55


# combine data

In [49]:
# generate key for raw data
books_before['penulis'] = (books_before['penulis'].str.lower())
books_before['judul_buku'] = (books_before['judul_buku'].str.lower())
cols = ['judul_buku', 'penulis', 'isbn_issn']
books_before['key'] = (
    books_before[cols]
    .fillna('')
    .astype(str)
    .agg('|'.join, axis=1)
)
show(books_before[['key', 'judul_buku', 'penulis', 'isbn_issn']])

Loading ITables v2.7.0 from the internet... (need help?)


In [50]:
combined_df = books_before.merge(
    enrich_df[['key', 'confidence', 'volume_info']], 
    on='key', 
    how='left'
)
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29770 entries, 0 to 29769
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   kode_buku      29134 non-null  str    
 1   isbn_issn      26489 non-null  str    
 2   judul_buku     29770 non-null  str    
 3   subjudul       10587 non-null  str    
 4   penulis        28449 non-null  str    
 5   nama_belakang  28449 non-null  str    
 6   topik          29770 non-null  str    
 7   deskripsi      229 non-null    str    
 8   bahasa         29770 non-null  str    
 9   kategori       29770 non-null  str    
 10  tahun_terbit   29088 non-null  float64
 11  key            29770 non-null  str    
 12  confidence     26144 non-null  float64
 13  volume_info    20258 non-null  str    
dtypes: float64(2), str(12)
memory usage: 42.5 MB


In [51]:
show(enrich_df[enrich_df['raw_isbn'] == "0870943405"]['key'])

Loading ITables v2.7.0 from the internet... (need help?)


In [52]:
show(combined_df[combined_df['isbn_issn'] == "0870943405"]['key'])

Loading ITables v2.7.0 from the internet... (need help?)


# extract data from volume_info

In [53]:
def extract_json_data(v_info):
    title = v_info.get('title')
    subtitle = v_info.get('subtitle', '')
    authors = v_info.get('authors', [])
    description = v_info.get('description')

    isbn_list = v_info.get('industryIdentifiers', [])
    isbn_10 = None
    isbn_13 = None
    isbn_list = v_info.get('industryIdentifiers', [])
    for identifier in isbn_list:
        if identifier['type'] == 'ISBN_10':
            isbn_10 = identifier['identifier']
        elif identifier['type'] == 'ISBN_13':
            isbn_13 = identifier['identifier']

    categories = v_info.get('categories', [])
    language = v_info.get('language')
    image = v_info.get('imageLinks', {}).get('thumbnail')

    return {
        'title_enrich': title,
        'authors_enrich': authors,
        'description_enrich': description,
        'subtitle_enrich': subtitle,
        'isbn_10_enrich': isbn_10,
        'isbn_13_enrich': isbn_13,
        'categories_enrich': categories,
        'language_enrich': language,
        'image_url': image
    }

In [54]:
def extract_volume_info(df):

    expected_cols = [
        'title_enrich', 'authors_enrich', 'description_enrich',
        'subtitle_enrich', 'isbn_10_enrich', 'isbn_13_enrich',
        'categories_enrich', 'language_enrich', 'image_url'
    ]

    def parse_and_extract(val):
        if pd.isna(val) or val in ['', 'None', None]:
            return pd.Series({col: None for col in expected_cols})
        
        try:
            v_info = json.loads(val) if isinstance(val, str) else val
            data = extract_json_data(v_info)
            return pd.Series(data)
        except (json.JSONDecodeError, TypeError):
            return pd.Series({col: None for col in expected_cols})

    enriched_cols = df['volume_info'].apply(parse_and_extract)

    return pd.concat([df, enriched_cols], axis=1)

In [55]:
extracted_df = extract_volume_info(combined_df)
extracted_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29770 entries, 0 to 29769
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   kode_buku           29134 non-null  str    
 1   isbn_issn           26489 non-null  str    
 2   judul_buku          29770 non-null  str    
 3   subjudul            10587 non-null  str    
 4   penulis             28449 non-null  str    
 5   nama_belakang       28449 non-null  str    
 6   topik               29770 non-null  str    
 7   deskripsi           229 non-null    str    
 8   bahasa              29770 non-null  str    
 9   kategori            29770 non-null  str    
 10  tahun_terbit        29088 non-null  float64
 11  key                 29770 non-null  str    
 12  confidence          26144 non-null  float64
 13  volume_info         20258 non-null  str    
 14  title_enrich        20254 non-null  str    
 15  authors_enrich      20256 non-null  object 
 16  description_enr

In [56]:
extracted_df['bahasa'].value_counts()

bahasa
English      16749
Indonesia    13021
Name: count, dtype: int64

In [57]:
extracted_df['language_enrich'].value_counts()

language_enrich
en    14812
id     5319
ms       64
nl       15
de        8
ar        7
es        6
ia        6
jv        5
fr        4
ko        2
su        2
it        1
pl        1
hu        1
ml        1
Name: count, dtype: int64

In [58]:
language_map = {
    'en': 'English',
    'id': 'Indonesia',
    'ms': 'Malay',
    'nl': 'Dutch',
    'de': 'German',
    'ar': 'Arabic',
    'ia': 'Interlingua',
    'es': 'Spanish',
    'ko': 'Korean',
    'fr': 'French',
    'it': 'Italian',
    'jv': 'Javanese'
}

extracted_df['language_enrich'] = extracted_df['language_enrich'].map(language_map)

In [59]:
extracted_df.sample(5)

,kode_buku,isbn_issn,judul_buku,subjudul,penulis,nama_belakang,topik,deskripsi,bahasa,kategori,...,volume_info,title_enrich,authors_enrich,description_enrich,subtitle_enrich,isbn_10_enrich,isbn_13_enrich,categories_enrich,language_enrich,image_url
726,"['90086703', '90086701', '90086702']",0072312653,production operations analysis,NaN,steven nahmias,nahmias,['production management'],NaN,Indonesia,Ilmu Pengetahuan Terapan/Teknologi,...,"{""title"": ""Production and Operations Analysis""...",Production and Operations Analysis,[Steven Nahmias],This text provides a survey of the analytical ...,,0072312653,9780072312652,[Administración de la producción],English,http://books.google.com/books/content?id=-fKvA...
28635,['923021923'],NaN,peraturan daerah provinsi jawa timur nomor 4 t...,NaN,indonesia. peraturan pemerintah,pemerintah,['archives'],NaN,Indonesia,Karya Umum,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN
11812,['605537'],9781779561916,heat transfer principles,NaN,artem shlyakhov marie-magdeleine,marie-magdeleine,[],NaN,English,Ilmu Pengetahuan Murni,...,NaN,NaN,None,NaN,NaN,NaN,NaN,None,NaN,NaN
3623,"['91335801', '91335802']",9780128121658,pollution control and resource recovery,municipal solid wastes incinerationl bottom as...,zhao youcai,youcai,['sewage sludge -- management'],NaN,English,Ilmu Pengetahuan Terapan/Teknologi,...,"{""title"": ""Pollution Control and Resource Reco...",Pollution Control and Resource Recovery,[Zhao Youcai],Pollution Control and Resource Recovery: Munic...,Municipal Solid Wastes Incineration: Bottom As...,0128121653,9780128121658,[Technology & Engineering],English,http://books.google.com/books/content?id=tK1Tv...
12845,"['613177', '609372']",9781898823599,japanese studies in britain,a survey and history,"hugh cortazzi, peter kornicki",cortazzi,[],NaN,English,Ilmu Sosial,...,"{""title"": ""Japanese Studies in Britain"", ""subt...",Japanese Studies in Britain,"[Hugh Cortazzi, Peter Francis Kornicki]",NaN,A Survey and History,1898823596,9781898823599,[EDUCATION],English,NaN


In [60]:
extracted_df['kode_buku'].dtype

<StringDtype(na_value=nan)>

In [61]:
import ast
import numpy as np

def str_to_list(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    
    if isinstance(val, list):
        return val
    
    if isinstance(val, (np.ndarray, pd.Series)):
        return val.tolist()
    
    if isinstance(val, str):
        val = val.strip()
        if val.startswith('[') and val.endswith(']'):
            try:
                return ast.literal_eval(val)
            except:
                return [val]  
        else:
            return [val]
    
    return [val]


In [62]:
extracted_df['kode_buku'] = extracted_df['kode_buku'].apply(str_to_list)
extracted_df['topik'] = extracted_df['topik'].apply(str_to_list)
extracted_df['authors_enrich'] = extracted_df['authors_enrich'].apply(str_to_list)
extracted_df['categories_enrich'] = extracted_df['categories_enrich'].apply(str_to_list)

In [63]:
extracted_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29770 entries, 0 to 29769
Data columns (total 23 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   kode_buku           29770 non-null  object 
 1   isbn_issn           26489 non-null  str    
 2   judul_buku          29770 non-null  str    
 3   subjudul            10587 non-null  str    
 4   penulis             28449 non-null  str    
 5   nama_belakang       28449 non-null  str    
 6   topik               29770 non-null  object 
 7   deskripsi           229 non-null    str    
 8   bahasa              29770 non-null  str    
 9   kategori            29770 non-null  str    
 10  tahun_terbit        29088 non-null  float64
 11  key                 29770 non-null  str    
 12  confidence          26144 non-null  float64
 13  volume_info         20258 non-null  str    
 14  title_enrich        20254 non-null  str    
 15  authors_enrich      29770 non-null  object 
 16  description_enr

In [64]:
extracted_df[extracted_df['judul_buku'] == 'administrasi pemerintahan daerah']

,kode_buku,isbn_issn,judul_buku,subjudul,penulis,nama_belakang,topik,deskripsi,bahasa,kategori,...,volume_info,title_enrich,authors_enrich,description_enrich,subtitle_enrich,isbn_10_enrich,isbn_13_enrich,categories_enrich,language_enrich,image_url
17895,"[91243501, 91243502, 91243503]",9789790114203,administrasi pemerintahan daerah,NaN,hanif nurcholis,nurcholis,[public administration],NaN,Indonesia,Ilmu Sosial,...,"{""title"": ""Administrasi pemerintahan daerah"",\...",NaN,[],NaN,NaN,NaN,NaN,[],NaN,NaN


## compare data

In [65]:
import re

def clean_isbn(x):
    if pd.isna(x):
        return None
    return str(x).replace('-', '').split('.')[0].strip()

def merge_title(row):
    old_title = str(row['judul_buku']) if pd.notna(row['judul_buku']) else ""
    new_title = str(row['title_enrich']) if pd.notna(row['title_enrich']) else ""
    
    if not new_title:
        return old_title
    
    vol_pattern = r"(jilid|jld|vol|volume|part|bagian|ke-?)\s?\d+"
    match_old = re.search(vol_pattern, old_title, re.IGNORECASE)
    
    if match_old and not re.search(vol_pattern, new_title, re.IGNORECASE):
        return f"{new_title} ({match_old.group(0)})"
    
    return new_title

def combine_enrich(df):
    df['isbn_clean'] = df['isbn_issn'].apply(clean_isbn)
    df['isbn10_clean'] = df['isbn_10_enrich'].apply(clean_isbn)
    df['isbn13_clean'] = df['isbn_13_enrich'].apply(clean_isbn)

    df['judul_buku'] = df.apply(merge_title, axis=1)

    df['penulis'] = df.apply(
        lambda row: ', '.join(row['authors_enrich'])
        if isinstance(row['authors_enrich'], list) and (
            pd.isna(row['penulis']) or
            row['penulis'] not in ', '.join(row['authors_enrich'])
        )
        else row['penulis'],
        axis=1
    )

    df['deskripsi'] = df['deskripsi'].fillna(df['description_enrich'])

    df['subjudul'] = df['subjudul'].fillna(df['subtitle_enrich'])

    df['topik'] = df.apply(
        lambda row: ', '.join(row['categories_enrich'])
        if isinstance(row['categories_enrich'], list) and (
            (pd.isna(row['topik']) if not isinstance(row['topik'], list) else len(row['topik']) == 0)
        )
        else (', '.join(row['topik']) if isinstance(row['topik'], list) else row['topik']),
        axis=1
    )

    df['bahasa'] = df.apply(
        lambda row: row['language_enrich']
        if pd.notna(row['language_enrich']) and row['bahasa'] != row['language_enrich']
        else row['bahasa'],
        axis=1
    )

    def update_isbn(row):
        if not row['isbn_clean']:
            return row['isbn10_clean'] or row['isbn13_clean']

        if row['isbn_clean'] not in [row['isbn10_clean'], row['isbn13_clean']]:
            return row['isbn10_clean'] or row['isbn13_clean']

        return row['isbn_issn']

    df['isbn_issn'] = df.apply(update_isbn, axis=1)
    
    return df

In [66]:
combined_enrich = combine_enrich(extracted_df)
combined_enrich.info()

<class 'pandas.DataFrame'>
RangeIndex: 29770 entries, 0 to 29769
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   kode_buku           29770 non-null  object 
 1   isbn_issn           18066 non-null  str    
 2   judul_buku          29770 non-null  str    
 3   subjudul            22696 non-null  str    
 4   penulis             29770 non-null  str    
 5   nama_belakang       28449 non-null  str    
 6   topik               29770 non-null  str    
 7   deskripsi           14233 non-null  str    
 8   bahasa              29770 non-null  str    
 9   kategori            29770 non-null  str    
 10  tahun_terbit        29088 non-null  float64
 11  key                 29770 non-null  str    
 12  confidence          26144 non-null  float64
 13  volume_info         20258 non-null  str    
 14  title_enrich        20254 non-null  str    
 15  authors_enrich      29770 non-null  object 
 16  description_enr

In [67]:
combined_enrich.drop(columns=['isbn_clean', 'title_enrich', 'isbn10_clean', 'isbn13_clean', 'authors_enrich','description_enrich', 'subtitle_enrich', 'categories_enrich', 'language_enrich', ], inplace=True)

In [68]:
show(combined_enrich)

Loading ITables v2.7.0 from the internet... (need help?)


In [69]:
combined_enrich['subjudul'] = (combined_enrich['subjudul'].fillna(''))

In [70]:
import numpy as np

combined_enrich['judul_full'] = np.where(
    combined_enrich['subjudul'] != '',
    combined_enrich['judul_buku'] + ' : ' + combined_enrich['subjudul'],
    combined_enrich['judul_buku']
)

In [71]:
combined_enrich[['judul_buku', 'subjudul', 'penulis', 'judul_full']].sample(5)

,judul_buku,subjudul,penulis,judul_full
10042,Handbook of Good Psychiatric Management for Bo...,,"Lois W. Choi-Kain, Hilary Connery",Handbook of Good Psychiatric Management for Bo...
3480,Operations and Supply Chain Management,,"F. Robert Jacobs, Richard B. Chase",Operations and Supply Chain Management
14997,Media massa dan konstruksi realitas,,Kun Wazis,Media massa dan konstruksi realitas
2917,Introduction to Modern Business,,"Vernon A. Musselman, John Harold Jackson",Introduction to Modern Business
4736,Edinburgh Companion to Samuel Beckett and the ...,,S.E. Gontarski,Edinburgh Companion to Samuel Beckett and the ...


In [72]:
combined_enrich[combined_enrich['judul_full'] == 'administrasi pemerintahan daerah']

,kode_buku,isbn_issn,judul_buku,subjudul,penulis,nama_belakang,topik,deskripsi,bahasa,kategori,tahun_terbit,key,confidence,volume_info,isbn_10_enrich,isbn_13_enrich,image_url,judul_full
17895,"[91243501, 91243502, 91243503]",NaN,administrasi pemerintahan daerah,,,nurcholis,public administration,NaN,Indonesia,Ilmu Sosial,2017.0,administrasi pemerintahan daerah|hanif nurchol...,1.0,"{""title"": ""Administrasi pemerintahan daerah"",\...",NaN,NaN,NaN,administrasi pemerintahan daerah


# Duplicate Books

In [73]:
combined_enrich.duplicated(subset='judul_full', keep=False).sum()

np.int64(2645)

In [74]:
combined_enrich.duplicated(subset=['judul_full', 'penulis'], keep=False).sum()

np.int64(2155)

In [75]:
show(combined_enrich[combined_enrich.duplicated(subset='judul_full', keep=False)])

Loading ITables v2.7.0 from the internet... (need help?)


In [76]:
combined_enrich.duplicated(subset='key', keep=False).sum()

np.int64(0)

In [77]:
def combine_book_data1(df):
    df['tahun_terbit'] = pd.to_numeric(df['tahun_terbit'], errors='coerce')
    
    df = df.sort_values(by=['tahun_terbit'], ascending=True)

    def aggregate_books(series):
        col_name = series.name
        
        clean_values = series.dropna().astype(str).unique()
        clean_values = [v for v in clean_values if v.lower() not in ['nan', 'none', '', '[]']]

        if col_name in ['kode_buku', 'topik']:
            all_items = []
            for val in clean_values:
                if val.startswith('[') and val.endswith(']'):
                    try:
                        # Handle string-represented lists
                        items = json.loads(val.replace("'", '"'))
                        all_items.extend(items if isinstance(items, list) else [items])
                    except:
                        all_items.append(val)
                else:
                    all_items.append(val)
            return list(set(all_items))

        if col_name == 'volume_info':
            if not clean_values: return None
            return max(clean_values, key=len)

        return clean_values[0] if clean_values else None

    grouped = df.groupby(['judul_full', 'penulis'], as_index=False).agg(aggregate_books)

    return grouped

In [78]:
duplicate_clean = combine_book_data1(combined_enrich)
duplicate_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 28451 entries, 0 to 28450
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   judul_full      28451 non-null  str   
 1   penulis         28451 non-null  str   
 2   kode_buku       28451 non-null  object
 3   isbn_issn       17459 non-null  str   
 4   judul_buku      28451 non-null  str   
 5   subjudul        12297 non-null  str   
 6   nama_belakang   27160 non-null  str   
 7   topik           28451 non-null  object
 8   deskripsi       13985 non-null  str   
 9   bahasa          28451 non-null  str   
 10  kategori        28451 non-null  str   
 11  tahun_terbit    27797 non-null  str   
 12  key             28451 non-null  str   
 13  confidence      24910 non-null  str   
 14  volume_info     19604 non-null  str   
 15  isbn_10_enrich  17457 non-null  str   
 16  isbn_13_enrich  17463 non-null  str   
 17  image_url       13250 non-null  str   
dtypes: object(2), str

In [79]:
duplicate_clean[duplicate_clean['judul_full'] == 'administrasi pemerintahan daerah']

,judul_full,penulis,kode_buku,isbn_issn,judul_buku,subjudul,nama_belakang,topik,deskripsi,bahasa,kategori,tahun_terbit,key,confidence,volume_info,isbn_10_enrich,isbn_13_enrich,image_url
19817,administrasi pemerintahan daerah,,"[91243501, 91243502, 91243503]",NaN,administrasi pemerintahan daerah,NaN,nurcholis,[public administration],NaN,Indonesia,Ilmu Sosial,2017.0,administrasi pemerintahan daerah|hanif nurchol...,1.0,"{""title"": ""Administrasi pemerintahan daerah"",\...",NaN,NaN,NaN


In [80]:
duplicate_clean.sample(5)

,judul_full,penulis,kode_buku,isbn_issn,judul_buku,subjudul,nama_belakang,topik,deskripsi,bahasa,kategori,tahun_terbit,key,confidence,volume_info,isbn_10_enrich,isbn_13_enrich,image_url
28295,urbanisasi dan kemiskinan : di dunia ke tiga,,"[90714101, 90714102]",NaN,urbanisasi dan kemiskinan,di dunia ke tiga,gilbert,[urbanization],NaN,Indonesia,Ilmu Sosial,1996.0,urbanisasi dan kemiskinan|alan gilbert|979812054,0.0,NaN,NaN,NaN,NaN
21183,china's economy 2010,,[616937],NaN,china's economy 2010,NaN,the institute of economic research,[],NaN,English,Ilmu Sosial,2012.0,china's economy 2010|renmin university of chin...,0.0,NaN,NaN,NaN,NaN
3896,Cyber Peace : Charting a Path Toward a Sustain...,"Scott J. Shackelford, Frederick Douzet, Christ...",[600136],110894941X,Cyber Peace,"Charting a Path Toward a Sustainable, Stable, ...",shackelford,"[international relations, hukum, pengantar, pu...",The international community is too often focus...,English,Lainnya,2022.0,"cyber peace|scott j. shackelford, frederick do...",1.0,"{""title"": ""Cyber Peace"", ""subtitle"": ""Charting...",110894941X,9781108949415,NaN
24010,manajemen pemasaran sudut pandang asia,,"[91031402, 91031401]",NaN,manajemen pemasaran sudut pandang asia,NaN,kotler,[digital marketing],NaN,Indonesia,Ilmu Pengetahuan Terapan/Teknologi,2004.0,manajemen pemasaran sudut pandang asia|philip ...,0.0,NaN,NaN,NaN,NaN
5071,Ekonomi Islam,,[],9789797691714,Ekonomi Islam,NaN,islam,[islam and economics],On principles of economy according to Islam.,Indonesia,Agama,2008.0,ekonomi islam|pusat pengkajian dan pengembanga...,1.0,"{""title"": ""Ekonomi Islam"", ""publishedDate"": ""2...",9797691713,9789797691714,NaN


In [81]:
duplicate_clean.duplicated(subset='judul_full', keep=False).sum()

np.int64(535)

In [82]:
show(duplicate_clean[duplicate_clean.duplicated(subset='judul_full', keep=False)])

Loading ITables v2.7.0 from the internet... (need help?)


In [83]:
import re

def normalize_author(name):
    if not name or pd.isna(name): return ""
    parts = sorted(re.sub(r'[^\w\s]', '', str(name).lower()).split())
    return " ".join(parts)

def combine_book_data2(df):
    df['author_norm'] = df['penulis'].apply(normalize_author)
    df['title_norm'] = df['judul_full'].str.lower().str.strip()
    
    df['tahun_terbit'] = pd.to_numeric(df['tahun_terbit'], errors='coerce')
    
    df = df.sort_values(by=['tahun_terbit'], ascending=True, na_position='last')

    def aggregate_books(series):
        col_name = series.name
        
        clean_values = series.dropna().astype(str).unique()
        clean_values = [v for v in clean_values if v.lower() not in ['nan', 'none', '', '[]']]
        
        if not clean_values:
            return None

        if col_name in ['kode_buku', 'topik']:
            all_items = []
            for val in clean_values:
                # Cek jika string berbentuk list 
                if str(val).startswith('[') and str(val).endswith(']'):
                    try:
                        cleaned_json = str(val).replace("'", '"')
                        items = json.loads(cleaned_json)
                        if isinstance(items, list):
                            all_items.extend(items)
                        else:
                            all_items.append(items)
                    except:
                        content = str(val).strip('[]').replace("'", "").split(',')
                        all_items.extend([c.strip() for c in content if c.strip()])
                else:
                    all_items.append(val)
            # Kembalikan list unik, buang duplikat
            return list(set(all_items))

        if col_name == 'volume_info':
            return max(clean_values, key=len)

        return clean_values[0]

    grouped = df.groupby(['title_norm', 'author_norm'], as_index=False).agg(aggregate_books)

    grouped = grouped.drop(columns=['author_norm', 'title_norm'])
    
    return grouped

In [84]:
duplicate_clean2 = combine_book_data2(duplicate_clean)
duplicate_clean2.info()

<class 'pandas.DataFrame'>
RangeIndex: 28400 entries, 0 to 28399
Data columns (total 18 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   judul_full      28400 non-null  str   
 1   penulis         18597 non-null  str   
 2   kode_buku       27912 non-null  object
 3   isbn_issn       17437 non-null  str   
 4   judul_buku      28400 non-null  str   
 5   subjudul        12274 non-null  str   
 6   nama_belakang   27114 non-null  str   
 7   topik           24439 non-null  object
 8   deskripsi       13970 non-null  str   
 9   bahasa          28400 non-null  str   
 10  kategori        28400 non-null  str   
 11  tahun_terbit    27748 non-null  str   
 12  key             28400 non-null  str   
 13  confidence      24864 non-null  str   
 14  volume_info     19575 non-null  str   
 15  isbn_10_enrich  17435 non-null  str   
 16  isbn_13_enrich  17441 non-null  str   
 17  image_url       13236 non-null  str   
dtypes: object(2), str

In [85]:
duplicate_clean2[duplicate_clean2['judul_full'] == 'administrasi pemerintahan daerah']

,judul_full,penulis,kode_buku,isbn_issn,judul_buku,subjudul,nama_belakang,topik,deskripsi,bahasa,kategori,tahun_terbit,key,confidence,volume_info,isbn_10_enrich,isbn_13_enrich,image_url
620,administrasi pemerintahan daerah,NaN,"[91243501, 91243502, 91243503]",NaN,administrasi pemerintahan daerah,NaN,nurcholis,[public administration],NaN,Indonesia,Ilmu Sosial,2017.0,administrasi pemerintahan daerah|hanif nurchol...,1.0,"{""title"": ""Administrasi pemerintahan daerah"",\...",NaN,NaN,NaN


In [86]:
show(duplicate_clean2[duplicate_clean2.duplicated(subset='judul_full', keep=False)])

Loading ITables v2.7.0 from the internet... (need help?)


# engineering book id

In [87]:
id_book_df = combined_enrich.copy()

id_book_df['book_id'] = id_book_df.reset_index().index + 1  # start from 1
id_book_df['book_id'] = id_book_df['book_id'].apply(lambda x: f"B{x:06d}")

In [88]:
id_book_df.sample(5)

,kode_buku,isbn_issn,judul_buku,subjudul,penulis,nama_belakang,topik,deskripsi,bahasa,kategori,tahun_terbit,key,confidence,volume_info,isbn_10_enrich,isbn_13_enrich,image_url,judul_full,book_id
4537,[619526],9780748646364,Scottish Ethnicity and the Making of New Zeala...,,Tanja Bueltmann,bueltmann,Social Science,The Scots accounted for around a quarter of al...,English,Lainnya,2011.0,scottish ethnicity and the making of new zeala...,0.963504,"{""title"": ""Scottish Ethnicity and the Making o...",0748646361,9780748646364,http://books.google.com/books/content?id=pS2rB...,Scottish Ethnicity and the Making of New Zeala...,B004538
21870,"[0490083801, 0490231101]",NaN,strategi mengeoa pubic reatins organisasi,,,suryadi,strategi,NaN,Indonesia,Ilmu Pengetahuan Terapan/Teknologi,2007.0,strategi mengeoa pubic reatins organisasi|drs....,0.000000,NaN,NaN,NaN,NaN,strategi mengeoa pubic reatins organisasi,B021871
28367,"[607482, 605981, 607825, 606324]",NaN,the territory of wei-hai-wei,a descriptive guide and handbook,,mitford,,NaN,English,Lainnya,1902.0,the territory of wei-hai-wei|c.e. bruce mitford|,NaN,NaN,NaN,NaN,NaN,the territory of wei-hai-wei : a descriptive g...,B028368
23367,"[FEB9000233, FEB9000234]",NaN,ekonomi publik,,,m.,ekonomi,NaN,Indonesia,Ilmu Sosial,1993.0,ekonomi publik|dr.guritno m.|9795032933,0.000000,NaN,NaN,NaN,NaN,ekonomi publik,B023368
27702,"[607288, 605787]",NaN,typical women of china,,,hsiang,,NaN,English,Lainnya,1899.0,typical women of china|liu hsiang|,NaN,NaN,NaN,NaN,NaN,typical women of china,B027703


In [89]:
id_book_df['kode_buku'].dtype

dtype('O')

In [90]:
id_book_df.sample(5)

,kode_buku,isbn_issn,judul_buku,subjudul,penulis,nama_belakang,topik,deskripsi,bahasa,kategori,tahun_terbit,key,confidence,volume_info,isbn_10_enrich,isbn_13_enrich,image_url,judul_full,book_id
10564,[604275],9781773618920,Advances in dairy research,,,kartan,,NaN,English,Ilmu Pengetahuan Terapan/Teknologi,2019.0,advances in dairy research|preethi kartan|9781...,1.0,"{""title"": ""Advances in dairy research"", ""publi...",177361892X,9781773618920,NaN,Advances in dairy research,B010565
21038,[0490221001],9792061428,Reinvensi BUMN,,Djokosantoso Moeljono,djokosantoso,Corporate culture,NaN,Indonesia,Lainnya,2004.0,reinvensi bumn|dr.moejono djokosantoso|9792061428,1.0,"{""title"": ""Reinvensi BUMN"", ""authors"": [""Djoko...",9792061428,9789792061420,http://books.google.com/books/content?id=2Q-Tp...,Reinvensi BUMN,B021039
27103,[90760601],NaN,sistem nilai manajer indonesia,tinjauan kritis berdasar penelitian,,danandjaja,managerial system,NaN,Indonesia,Ilmu Pengetahuan Terapan/Teknologi,1986.0,sistem nilai manajer indonesia|andreas a. dana...,NaN,NaN,NaN,NaN,NaN,sistem nilai manajer indonesia : tinjauan krit...,B027104
6491,"[614802, 611306]",9781447321873,Revisiting Moral Panics,,"Viviene E. Cree, Gary Clapton, Mark Smith",cree,Social Science,We live in a world that is increasingly charac...,English,Ilmu Sosial,2015.0,"revisiting moral panics|viviene e. cree, gary ...",1.0,"{""title"": ""Revisiting Moral Panics"", ""authors""...",1447321871,9781447321873,http://books.google.com/books/content?id=ZwVpD...,Revisiting Moral Panics,B006492
22637,[0490238401],979421874X,Otonomi daerah dan daerah otonom,,A. W. Widjaya,sidjaja,otonomi,On provincial autonomy and local government fi...,Indonesia,Ilmu Sosial,2002.0,"metode peneitian kuantitatif, kuaitatif dan r&...",1.0,"{""title"": ""Otonomi daerah dan daerah otonom"", ...",979421874X,9789794218747,NaN,Otonomi daerah dan daerah otonom,B022638


# enginer book id in trans data

In [91]:
trans_data = pd.read_csv('./data/processed/trans_after_cleaning.csv')
trans_data.sample(3)

,ID Anggota,Nama Anggota,Kode Eksemplar,Judul,Tanggal Pinjam,tahun_pinjam,bulan_pinjam,bulan_tahun_pinjam
3729,23013010159,Shafa Rizkika Hamidah,91358702,Auditing dan asuransi : pemeriksaan akuntansi ...,2025-02-17,2025,2,2025-02
4866,22013010357,Devana salsabila,91554703,Akuntansi Keuangan Daerah Berbasis Akrual,2025-09-22,2025,9,2025-09
3494,21011010144,Yuda Shagy Rahmanda Putra,90830702,Ilmu makroekonomi,2025-01-09,2025,1,2025-01


In [92]:
trans_data['Kode Eksemplar'] = trans_data['Kode Eksemplar'].astype(str).str.strip()

In [93]:
id_book_df['judul_full'] = id_book_df.apply(
    lambda x: f"{x['judul_buku']} {x['subjudul']}" if pd.notna(x['subjudul']) and str(x['subjudul']).strip() != '' 
    else x['judul_buku'], 
    axis=1
)

In [94]:
lookup_df = id_book_df[['kode_buku', 'judul_full', 'isbn_issn', 'book_id']].copy()
lookup_df

,kode_buku,judul_full,isbn_issn,book_id
0,[FEB90000172],business & management,NaN,B000001
1,[FEB90002446],business dan management,NaN,B000002
2,"[90572001, 90572002, 90572003, 90572004, 90572...",Preparing for the Twenty-first Century,0002157055,B000003
3,[91131601],Grammar in Use Self-study Reference and Practi...,0007163460,B000004
4,"[90006301, 90006302, 90006303]",The Illustrator 9 Wow! Book,0201704536,B000005
...,...,...,...,...
29765,[90345601],international bisnis management text and cases,NaN,B029766
29766,[91664701],who's afraid of the big bad dragon? why china ...,NaN,B029767
29767,[90367701],kamus istilah-istilah biologi,NaN,B029768
29768,"[90277401, 90277402, 90277403]",equalization of educational opportunities for ...,NaN,B029769


In [95]:
lookup_df = lookup_df.explode('kode_buku')
show(lookup_df)

Loading ITables v2.7.0 from the internet... (need help?)


In [96]:
lookup_df.info()

<class 'pandas.DataFrame'>
Index: 67073 entries, 0 to 29769
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   kode_buku   66437 non-null  str  
 1   judul_full  67073 non-null  str  
 2   isbn_issn   32644 non-null  str  
 3   book_id     67073 non-null  str  
dtypes: str(4)
memory usage: 7.2 MB


In [97]:
lookup_df['kode_buku'] = lookup_df['kode_buku'].astype(str).str.strip()
show(lookup_df[lookup_df.duplicated(subset='book_id', keep=False)])

Loading ITables v2.7.0 from the internet... (need help?)


In [98]:
from thefuzz import process, fuzz
from tqdm import tqdm
import re

tqdm.pandas()

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text)          # normalize spaces
    return text

def enrich_transactions_with_fuzzy(trans_df, master_df, threshold=85):

    master_df = master_df.copy()
    master_df['judul_clean'] = master_df['judul_full'].apply(clean_text)

    title_to_bookid = dict(
        zip(master_df['judul_clean'], master_df['book_id'])
    )

    master_titles = list(title_to_bookid.keys())

    master_exploded = master_df.explode('kode_buku')[['kode_buku', 'book_id']]
    master_exploded['kode_buku'] = master_exploded['kode_buku'].astype(str).str.strip()

    merged = trans_df.merge(
        master_exploded,
        left_on='Kode Eksemplar',
        right_on='kode_buku',
        how='left'
    )

    unmatched_mask = merged['book_id'].isna()

    if unmatched_mask.sum() == 0:
        return merged.drop(columns=['kode_buku'], errors='ignore')

    print(f"Fuzzy matching {unmatched_mask.sum()} rows...")

    fuzzy_cache = {}

    def fuzzy_get_bookid(title):
        title_clean = clean_text(title)

        if not title_clean:
            return None

        if title_clean in fuzzy_cache:
            return fuzzy_cache[title_clean]

        # use token_set_ratio (better for reordered words)
        best_match = process.extractOne(
            title_clean,
            master_titles,
            scorer=fuzz.token_set_ratio
        )

        if best_match and best_match[1] >= threshold:
            result = title_to_bookid[best_match[0]]
        else:
            result = None

        fuzzy_cache[title_clean] = result
        return result

    merged.loc[unmatched_mask, 'book_id'] = (
        merged.loc[unmatched_mask, 'Judul']
        .progress_apply(fuzzy_get_bookid)
    )

    return merged.drop(columns=['kode_buku'], errors='ignore')

In [99]:
trans_engineered = enrich_transactions_with_fuzzy(trans_data, lookup_df, threshold=80)

Fuzzy matching 22 rows...


100%|██████████| 22/22 [00:00<00:00, 29.43it/s]


In [100]:
trans_engineered.info()

<class 'pandas.DataFrame'>
RangeIndex: 5372 entries, 0 to 5371
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ID Anggota          5372 non-null   int64
 1   Nama Anggota        5372 non-null   str  
 2   Kode Eksemplar      5372 non-null   str  
 3   Judul               5372 non-null   str  
 4   Tanggal Pinjam      5372 non-null   str  
 5   tahun_pinjam        5372 non-null   int64
 6   bulan_pinjam        5372 non-null   int64
 7   bulan_tahun_pinjam  5372 non-null   str  
 8   book_id             5372 non-null   str  
dtypes: int64(3), str(6)
memory usage: 873.4 KB


In [101]:
trans_engineered.sample(5)

,ID Anggota,Nama Anggota,Kode Eksemplar,Judul,Tanggal Pinjam,tahun_pinjam,bulan_pinjam,bulan_tahun_pinjam,book_id
4692,22013010294,WIDYA FRAHESTIKA,91554701,Akuntansi Keuangan Daerah Berbasis Akrual,2025-09-09,2025,9,2025-09,B018193
783,23042010098,Aprilia mariyam,91286401,"Manajemen Perilaku Organisasi, Edisi Revisi",2023-12-14,2023,12,2023-12,B021532
2500,24012010357,nabila amalia syahputri,91093902,Pengantar Psikologi Ksehatan Islam,2024-09-04,2024,9,2024-09,B020155
690,21013010123,HENY SULISTIANA,91176106,"Akuntansi lanjutan (Advanced Accounting ), jil...",2023-12-05,2023,12,2023-12,B018294
3689,23011010086,SETIA DWI NINGTIAS,91349201,PERDAGANGAN INTERNASIONAL : Kupas Tuntas Prose...,2025-02-13,2025,2,2025-02,B014324


In [102]:
show(trans_engineered[trans_engineered['book_id'].isnull()])

Loading ITables v2.7.0 from the internet... (need help?)


In [103]:
id_book_df[id_book_df['judul_buku'].str.contains('Ekonofisika', case=False, na=False)]

,kode_buku,isbn_issn,judul_buku,subjudul,penulis,nama_belakang,topik,deskripsi,bahasa,kategori,tahun_terbit,key,confidence,volume_info,isbn_10_enrich,isbn_13_enrich,image_url,judul_full,book_id
23856,"[90916701, 90916702, 90916703, FEB90001987]",6233281024,Riset Operasi dan Ekonofisika (Operations Rese...,,"Drs. Suyadi Prawirosentono, M.M., M.B.A",prawirosentono,oparetion research,Organisasi bisnis dan publik di negara maju me...,Indonesia,Karya Umum,2005.0,riset operasi dan ekonofisika (operation rese...,1.0,"{""title"": ""Riset Operasi dan Ekonofisika (Oper...",6233281024,9786233281027,http://books.google.com/books/content?id=i5o_E...,Riset Operasi dan Ekonofisika (Operations Rese...,B023857


# Filter bds deskripsi word count

In [104]:
id_book_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29770 entries, 0 to 29769
Data columns (total 19 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   kode_buku       29770 non-null  object 
 1   isbn_issn       18066 non-null  str    
 2   judul_buku      29770 non-null  str    
 3   subjudul        29770 non-null  str    
 4   penulis         29770 non-null  str    
 5   nama_belakang   28449 non-null  str    
 6   topik           29770 non-null  str    
 7   deskripsi       14233 non-null  str    
 8   bahasa          29770 non-null  str    
 9   kategori        29770 non-null  str    
 10  tahun_terbit    29088 non-null  float64
 11  key             29770 non-null  str    
 12  confidence      26144 non-null  float64
 13  volume_info     20258 non-null  str    
 14  isbn_10_enrich  18064 non-null  str    
 15  isbn_13_enrich  18070 non-null  str    
 16  image_url       13407 non-null  str    
 17  judul_full      29770 non-null  str    
 1

In [105]:
id_book_df['deskripsi'].notna().sum()

np.int64(14233)

In [106]:
books_df_filtered = id_book_df.copy()

In [107]:
books_df_filtered['deskripsi_count'] = books_df_filtered['deskripsi'].apply(lambda x: len(str(x).split()) if pd.notna(x) else 0)
books_df_filtered['deskripsi_count'].describe()

count    29770.000000
mean        58.978939
std         93.680237
min          0.000000
25%          0.000000
50%          0.000000
75%        105.000000
max       1252.000000
Name: deskripsi_count, dtype: float64

In [108]:
bins = [-1, 0, 3, 10, 20, 50, 100, 1200]
labels = ['0 kata', '1-3 kata', '4-10 kata', '11-20 kata', '21-50 kata', '51-100 kata', '>100 kata']

books_df_filtered['deskripsi_count_category'] = pd.cut(
    books_df_filtered['deskripsi_count'],
    bins=bins,
    labels=labels
)

books_df_filtered['deskripsi_count_category'].value_counts()

deskripsi_count_category
0 kata         15537
>100 kata       7836
51-100 kata     2648
21-50 kata      1733
4-10 kata       1098
11-20 kata       836
1-3 kata          79
Name: count, dtype: int64

In [109]:
show(books_df_filtered[books_df_filtered['deskripsi_count_category'] == '1-3 kata'][['judul_full', 'deskripsi']])

Loading ITables v2.7.0 from the internet... (need help?)


In [110]:
show(books_df_filtered[books_df_filtered['deskripsi_count_category'] == '4-10 kata'][['judul_full', 'deskripsi']])

Loading ITables v2.7.0 from the internet... (need help?)


In [111]:
show(books_df_filtered[books_df_filtered['deskripsi_count_category'] == '11-20 kata'][['judul_full', 'deskripsi']])

Loading ITables v2.7.0 from the internet... (need help?)


hapus deskripsi di bawah 10 kata

In [112]:
books_df_filtered = books_df_filtered.copy()

In [113]:
mask = books_df_filtered['deskripsi_count'] <= 10
books_df_filtered.loc[mask, 'deskripsi'] = None
books_df_filtered.info()

<class 'pandas.DataFrame'>
RangeIndex: 29770 entries, 0 to 29769
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   kode_buku                 29770 non-null  object  
 1   isbn_issn                 18066 non-null  str     
 2   judul_buku                29770 non-null  str     
 3   subjudul                  29770 non-null  str     
 4   penulis                   29770 non-null  str     
 5   nama_belakang             28449 non-null  str     
 6   topik                     29770 non-null  str     
 7   deskripsi                 13056 non-null  str     
 8   bahasa                    29770 non-null  str     
 9   kategori                  29770 non-null  str     
 10  tahun_terbit              29088 non-null  float64 
 11  key                       29770 non-null  str     
 12  confidence                26144 non-null  float64 
 13  volume_info               20258 non-null  str     
 14  i

In [114]:
books_df_filtered['deskripsi_count_category'].value_counts()

deskripsi_count_category
0 kata         15537
>100 kata       7836
51-100 kata     2648
21-50 kata      1733
4-10 kata       1098
11-20 kata       836
1-3 kata          79
Name: count, dtype: int64

In [115]:
show(books_df_filtered[books_df_filtered['deskripsi_count_category'] == '1-3 kata'][['judul_full', 'deskripsi']])

Loading ITables v2.7.0 from the internet... (need help?)


# deskripsi untuk yang ada di trans data

In [116]:
borrowed_ids = set(trans_engineered['book_id'].dropna().unique())
borrowed_ids

{'B019956',
 'B021848',
 'B018110',
 'B028271',
 'B024315',
 'B023658',
 'B014032',
 'B000790',
 'B019853',
 'B027058',
 'B003855',
 'B012979',
 'B027632',
 'B021550',
 'B021840',
 'B022008',
 'B023320',
 'B018419',
 'B023088',
 'B028213',
 'B018086',
 'B014506',
 'B026124',
 'B022322',
 'B025404',
 'B023375',
 'B028283',
 'B025194',
 'B014487',
 'B020207',
 'B013715',
 'B019245',
 'B018446',
 'B013299',
 'B025994',
 'B025272',
 'B015065',
 'B025666',
 'B024197',
 'B000113',
 'B003245',
 'B005599',
 'B015097',
 'B022423',
 'B018245',
 'B005739',
 'B009022',
 'B021759',
 'B029679',
 'B019897',
 'B021156',
 'B027839',
 'B019968',
 'B021097',
 'B021163',
 'B023094',
 'B025167',
 'B024484',
 'B014162',
 'B019016',
 'B014692',
 'B025192',
 'B024266',
 'B018297',
 'B025553',
 'B021191',
 'B018244',
 'B025208',
 'B023082',
 'B014518',
 'B008984',
 'B015024',
 'B023858',
 'B026469',
 'B021856',
 'B024601',
 'B025968',
 'B001003',
 'B026414',
 'B018004',
 'B023309',
 'B022356',
 'B013151',
 'B0

In [118]:
len(borrowed_ids)

1960

In [119]:
books_borrowed = books_df_filtered[books_df_filtered['book_id'].isin(borrowed_ids)]
books_borrowed.info()

<class 'pandas.DataFrame'>
Index: 1960 entries, 9 to 29769
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   kode_buku                 1960 non-null   object  
 1   isbn_issn                 1030 non-null   str     
 2   judul_buku                1960 non-null   str     
 3   subjudul                  1960 non-null   str     
 4   penulis                   1960 non-null   str     
 5   nama_belakang             1902 non-null   str     
 6   topik                     1960 non-null   str     
 7   deskripsi                 497 non-null    str     
 8   bahasa                    1960 non-null   str     
 9   kategori                  1960 non-null   str     
 10  tahun_terbit              1877 non-null   float64 
 11  key                       1960 non-null   str     
 12  confidence                1732 non-null   float64 
 13  volume_info               1167 non-null   str     
 14  isbn_10

In [158]:
show(books_borrowed[books_borrowed['deskripsi'].notna()][['judul_buku', 'deskripsi', 'book_id']])

Loading ITables v2.7.0 from the internet... (need help?)


In [141]:
books_borrowed.loc[books_borrowed['deskripsi'].isna(), 'deskripsi'] = 'tidak ada deskripsi'
show(books_borrowed[books_borrowed['deskripsi'] == 'tidak ada deskripsi'][['judul_buku', 'deskripsi']])

Loading ITables v2.7.0 from the internet... (need help?)


In [142]:
desc_clean_df = books_df_filtered.copy()

mask = desc_clean_df['book_id'].isin(borrowed_ids) & desc_clean_df['deskripsi'].isna()
desc_clean_df.loc[mask, 'deskripsi'] = 'tidak ada deskripsi'

show(desc_clean_df[desc_clean_df['deskripsi'] == 'tidak ada deskripsi'])

Loading ITables v2.7.0 from the internet... (need help?)


In [143]:
desc_clean_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 29770 entries, 0 to 29769
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   kode_buku                 29770 non-null  object  
 1   isbn_issn                 18066 non-null  str     
 2   judul_buku                29770 non-null  str     
 3   subjudul                  29770 non-null  str     
 4   penulis                   29770 non-null  str     
 5   nama_belakang             28449 non-null  str     
 6   topik                     29770 non-null  str     
 7   deskripsi                 14519 non-null  str     
 8   bahasa                    29770 non-null  str     
 9   kategori                  29770 non-null  str     
 10  tahun_terbit              29088 non-null  float64 
 11  key                       29770 non-null  str     
 12  confidence                26144 non-null  float64 
 13  volume_info               20258 non-null  str     
 14  i

In [144]:
desc_clean_df.dropna(subset=['deskripsi'], inplace=True)
desc_clean_df.info()

<class 'pandas.DataFrame'>
Index: 14519 entries, 2 to 29769
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   kode_buku                 14519 non-null  object  
 1   isbn_issn                 12811 non-null  str     
 2   judul_buku                14519 non-null  str     
 3   subjudul                  14519 non-null  str     
 4   penulis                   14519 non-null  str     
 5   nama_belakang             13855 non-null  str     
 6   topik                     14519 non-null  str     
 7   deskripsi                 14519 non-null  str     
 8   bahasa                    14519 non-null  str     
 9   kategori                  14519 non-null  str     
 10  tahun_terbit              14322 non-null  float64 
 11  key                       14519 non-null  str     
 12  confidence                14274 non-null  float64 
 13  volume_info               13698 non-null  str     
 14  isbn_1

In [145]:
desc_clean_df.sample(5)

,kode_buku,isbn_issn,judul_buku,subjudul,penulis,nama_belakang,topik,deskripsi,bahasa,kategori,...,key,confidence,volume_info,isbn_10_enrich,isbn_13_enrich,image_url,judul_full,book_id,deskripsi_count,deskripsi_count_category
86,[90364201],NaN,Just-in-time Purchasing,,"Abdolhossein Ansari, B. Modarress",ansari,just - in - time systems,Ansari and Modarress have written the first fu...,English,Ilmu Pengetahuan Terapan/Teknologi,...,just in time purchasing|abdolhossein ansari|00...,0.913043,"{""title"": ""Just-in-time Purchasing"", ""authors""...",NaN,NaN,http://books.google.com/books/content?id=I8MSA...,Just-in-time Purchasing,B000087,27,21-50 kata
21227,"[90944201, 90944202]",9792501509,Potensi ekspor Jawa Timur ke [nama negara]: Sp...,,,auwalin,jawa timur-commerce-spanyol,Potential of commercial products for export fr...,Indonesia,Ilmu Sosial,...,potensi ekspor jawa timur ke spanyol|ilmiawan ...,1.000000,"{""title"": ""Potensi ekspor Jawa Timur ke [nama ...",9792501509,9789792501506,NaN,Potensi ekspor Jawa Timur ke [nama negara]: Sp...,B021228,14,11-20 kata
6805,"[615087, 611591]",9781447346227,Bordering Two Unions,northern ireland and brexit,"Sylvia de Mars, Colin Murray, Aoife O'Donoghue...",de mars,Law,Available Open Access under CC-BY-NC licence. ...,English,Ilmu Sosial,...,"bordering two unions|sylvia de mars, colin mur...",1.000000,"{""title"": ""Bordering Two Unions"", ""subtitle"": ...",144734622X,9781447346227,http://books.google.com/books/content?id=qXJoD...,Bordering Two Unions northern ireland and brexit,B006806,108,>100 kata
7288,[620073],9781474401890,Multiculturalism Rethought,"interpretations, diemmas and new directions : ...",Varun Uberoi,uberoi,Social Science,A selection of the leading theorists of multic...,English,Ilmu Sosial,...,"multiculturalism rethought|varun uberoi, tariq...",1.000000,"{""title"": ""Multiculturalism Rethought"", ""subti...",1474401899,9781474401890,http://books.google.com/books/content?id=EuckD...,"Multiculturalism Rethought interpretations, di...",B007289,26,21-50 kata
10087,[616984],1351611208,Internationalization of the RMB,2013 annual report,International Monetary Institute of the RUC,international monetary institute,Business & Economics,Renminbi (RMB) internationalization and the “O...,English,Ilmu Sosial,...,internationalization of the rmb|renmin univers...,1.000000,"{""title"": ""Internationalization of the RMB"", ""...",1351611208,9781351611206,http://books.google.com/books/content?id=ytqMD...,Internationalization of the RMB 2013 annual re...,B010088,92,51-100 kata


# Filter Bahasa

In [146]:
books_lang_filter = desc_clean_df.copy()

In [147]:
books_lang_filter.info()

<class 'pandas.DataFrame'>
Index: 14519 entries, 2 to 29769
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   kode_buku                 14519 non-null  object  
 1   isbn_issn                 12811 non-null  str     
 2   judul_buku                14519 non-null  str     
 3   subjudul                  14519 non-null  str     
 4   penulis                   14519 non-null  str     
 5   nama_belakang             13855 non-null  str     
 6   topik                     14519 non-null  str     
 7   deskripsi                 14519 non-null  str     
 8   bahasa                    14519 non-null  str     
 9   kategori                  14519 non-null  str     
 10  tahun_terbit              14322 non-null  float64 
 11  key                       14519 non-null  str     
 12  confidence                14274 non-null  float64 
 13  volume_info               13698 non-null  str     
 14  isbn_1

In [148]:
books_lang_filter['bahasa'].value_counts()

bahasa
English        11530
Indonesia       2946
Malay             20
Dutch             13
Interlingua        6
French             2
German             1
Korean             1
Name: count, dtype: int64

In [149]:
show(books_lang_filter[books_lang_filter['bahasa'] == 'Interlingua'])

Loading ITables v2.7.0 from the internet... (need help?)


Malay = masih indo sih, kayaknya miss detection
Dutch = dutch, dihapus
interlingua = indo
French =. 2 english, 1 literal french, B018145
german = german
korean = english


In [150]:
lang_mask = ['Dutch', 'French', 'German', 'Korean']
books_lang_filter = books_lang_filter[~books_lang_filter['bahasa'].isin(lang_mask)]
books_lang_filter['bahasa'].value_counts()

bahasa
English        11530
Indonesia       2946
Malay             20
Interlingua        6
Name: count, dtype: int64

In [151]:
lang_map = {
    'English': 'English',
    'Indonesia': 'Indonesia',
    'Malay': 'Indonesia',
    'Interlingua': 'Indonesia'
}
books_lang_filter['bahasa'] = books_lang_filter['bahasa'].map(lang_map)
books_lang_filter['bahasa'].value_counts()

bahasa
English      11530
Indonesia     2972
Name: count, dtype: int64

# Final Data

In [152]:
books_output = books_lang_filter.copy()
trans_output = trans_engineered.copy()

In [153]:
books_output.info()

<class 'pandas.DataFrame'>
Index: 14502 entries, 2 to 29769
Data columns (total 21 columns):
 #   Column                    Non-Null Count  Dtype   
---  ------                    --------------  -----   
 0   kode_buku                 14502 non-null  object  
 1   isbn_issn                 12794 non-null  str     
 2   judul_buku                14502 non-null  str     
 3   subjudul                  14502 non-null  str     
 4   penulis                   14502 non-null  str     
 5   nama_belakang             13838 non-null  str     
 6   topik                     14502 non-null  str     
 7   deskripsi                 14502 non-null  str     
 8   bahasa                    14502 non-null  str     
 9   kategori                  14502 non-null  str     
 10  tahun_terbit              14305 non-null  float64 
 11  key                       14502 non-null  str     
 12  confidence                14257 non-null  float64 
 13  volume_info               13681 non-null  str     
 14  isbn_1

In [154]:
books_output['deskripsi'].isna().sum()

np.int64(0)

In [155]:
books_output.to_csv('data/processed/books_enriched.csv', sep=';', index=False)

In [156]:
trans_output.info()

<class 'pandas.DataFrame'>
RangeIndex: 5372 entries, 0 to 5371
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   ID Anggota          5372 non-null   int64
 1   Nama Anggota        5372 non-null   str  
 2   Kode Eksemplar      5372 non-null   str  
 3   Judul               5372 non-null   str  
 4   Tanggal Pinjam      5372 non-null   str  
 5   tahun_pinjam        5372 non-null   int64
 6   bulan_pinjam        5372 non-null   int64
 7   bulan_tahun_pinjam  5372 non-null   str  
 8   book_id             5372 non-null   str  
dtypes: int64(3), str(6)
memory usage: 873.4 KB


In [157]:
trans_output.to_csv('data/processed/transactions_enriched.csv', sep=';', index=False)